In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

c:\Users\tyler\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
adult = pd.read_csv("../datasets/adult.csv")
adult.columns = [c.strip() for c in adult.columns]

# Clean string columns and missing markers
for col in adult.select_dtypes(include="object").columns:
    adult[col] = adult[col].astype(str).str.strip()
adult = adult.replace("?", np.nan)

# Binary target: income >50K -> 1, else 0
adult["income"] = adult["income"].map({">50K": 1, "<=50K": 0})

print(adult.shape)
adult.head()

(48842, 15)


C:\Users\tyler\AppData\Local\Temp\ipykernel_31508\2701957548.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in adult.select_dtypes(include="object").columns:


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0


In [3]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()

    # Transformation features
    x["log_capital_gain"] = np.log1p(x["capital-gain"].fillna(0))
    x["log_capital_loss"] = np.log1p(x["capital-loss"].fillna(0))

    # Interaction feature
    x["age_x_hours"] = x["age"].fillna(x["age"].median()) * x["hours-per-week"].fillna(x["hours-per-week"].median())

    # Grouped/binned feature
    x["age_group"] = pd.cut(
        x["age"],
        bins=[0, 25, 35, 45, 55, 100],
        labels=["18-25", "26-35", "36-45", "46-55", "56+"],
        include_lowest=True,
    ).astype(str)

    # Binary indicator features
    x["has_capital_gain"] = (x["capital-gain"].fillna(0) > 0).astype(int)
    x["high_hours_worker"] = (x["hours-per-week"].fillna(0) >= 45).astype(int)

    return x

X = adult.drop(columns=["income"])
y = adult["income"].astype(int)

X = add_engineered_features(X)
print("Feature matrix shape after engineering:", X.shape)
X.head()

Feature matrix shape after engineering: (48842, 20)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,log_capital_gain,log_capital_loss,age_x_hours,age_group,has_capital_gain,high_hours_worker
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0.000000,0.0,1000,18-25,0,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0.000000,0.0,1900,36-45,0,1
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,0.000000,0.0,1120,26-35,0,0
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,8.947546,0.0,1760,36-45,1,0
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0.000000,0.0,540,18-25,0,0


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_pipe_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_lr = ColumnTransformer([
    ("num", numeric_pipe_lr, num_features),
    ("cat", categorical_pipe, cat_features),
])

preprocess_rf = ColumnTransformer([
    ("num", numeric_pipe_rf, num_features),
    ("cat", categorical_pipe, cat_features),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (39073, 20), Test shape: (9769, 20)


C:\Users\tyler\AppData\Local\Temp\ipykernel_31508\2672952851.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(include=["object"]).columns.tolist()


In [5]:
# Model 1: Logistic Regression (linear model)
logreg_pipe = Pipeline([
    ("prep", preprocess_lr),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])

logreg_params = {
    "model__C": np.logspace(-2, 1, 8),
    "model__solver": ["liblinear", "lbfgs"],
    "model__class_weight": [None, "balanced"],
}

logreg_search = RandomizedSearchCV(
    estimator=logreg_pipe,
    param_distributions=logreg_params,
    n_iter=12,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    random_state=42,
)

logreg_search.fit(X_train, y_train)
print("Best Logistic Regression params:")
print(logreg_search.best_params_)
print("Best CV accuracy:", round(logreg_search.best_score_, 4))

Best Logistic Regression params:
{'model__solver': 'lbfgs', 'model__class_weight': None, 'model__C': np.float64(0.517947467923121)}
Best CV accuracy: 0.8579


In [6]:
# Model 2: Random Forest (non-linear tree model)
rf_pipe = Pipeline([
    ("prep", preprocess_rf),
    ("model", RandomForestClassifier(random_state=42)),
])

rf_params = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": [None, "balanced"],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_params,
    n_iter=16,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    random_state=42,
)

rf_search.fit(X_train, y_train)
print("Best Random Forest params:")
print(rf_search.best_params_)
print("Best CV accuracy:", round(rf_search.best_score_, 4))

Best Random Forest params:
{'model__n_estimators': 300, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_features': 'sqrt', 'model__max_depth': 20, 'model__class_weight': None}
Best CV accuracy: 0.8634


In [7]:
best_logreg = logreg_search.best_estimator_
best_rf = rf_search.best_estimator_

# Individual model performance
logreg_pred = best_logreg.predict(X_test)
rf_pred = best_rf.predict(X_test)

logreg_acc = accuracy_score(y_test, logreg_pred)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Logistic Regression Test Accuracy: {logreg_acc:.4f}")
print(classification_report(y_test, logreg_pred))
print("-" * 70)
print(f"Random Forest Test Accuracy: {rf_acc:.4f}")
print(classification_report(y_test, rf_pred))

# Task 4: Probability averaging ensemble
logreg_proba = best_logreg.predict_proba(X_test)[:, 1]
rf_proba = best_rf.predict_proba(X_test)[:, 1]

ensemble_proba = 0.5 * logreg_proba + 0.5 * rf_proba
ensemble_pred = (ensemble_proba >= 0.5).astype(int)
ensemble_acc = accuracy_score(y_test, ensemble_pred)

print("=" * 70)
print(f"Ensemble (avg probabilities) Test Accuracy: {ensemble_acc:.4f}")
print(classification_report(y_test, ensemble_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, ensemble_pred))

results = pd.DataFrame(
    {
        "model": ["Logistic Regression", "Random Forest", "Ensemble (avg proba)"],
        "cv_accuracy": [logreg_search.best_score_, rf_search.best_score_, np.nan],
        "test_accuracy": [logreg_acc, rf_acc, ensemble_acc],
    }
).sort_values("test_accuracy", ascending=False)

results

Logistic Regression Test Accuracy: 0.8588
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      7431
           1       0.76      0.60      0.67      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.77      0.79      9769
weighted avg       0.85      0.86      0.85      9769

----------------------------------------------------------------------
Random Forest Test Accuracy: 0.8655
              precision    recall  f1-score   support

           0       0.88      0.95      0.92      7431
           1       0.80      0.59      0.68      2338

    accuracy                           0.87      9769
   macro avg       0.84      0.77      0.80      9769
weighted avg       0.86      0.87      0.86      9769

Ensemble (avg probabilities) Test Accuracy: 0.8661
              precision    recall  f1-score   support

           0       0.88      0.95      0.92      7431
           1       0.79      0.60    

,model,cv_accuracy,test_accuracy
2,Ensemble (avg proba),NaN,0.866107
1,Random Forest,0.863409,0.865493
0,Logistic Regression,0.857932,0.858839


In [8]:
# Feature usefulness view from the tuned Random Forest
rf_model = best_rf.named_steps["model"]
rf_prep = best_rf.named_steps["prep"]
feature_names = rf_prep.get_feature_names_out()

importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": rf_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_df.head(15)

,feature,importance
37,cat__marital-status_Married-civ-spouse,0.114046
2,num__educational-num,0.097130
56,cat__relationship_Husband,0.082038
3,num__capital-gain,0.081913
6,num__log_capital_gain,0.081317
8,num__age_x_hours,0.063710
0,num__age,0.042149
39,cat__marital-status_Never-married,0.036904
5,num__hours-per-week,0.031300
4,num__capital-loss,0.029059


The most useful features were the marital/relationship indicators and income-signal numeric variables. In the tuned Random Forest importance table, top drivers included marital-status_Married-civ-spouse, educational-num, relationship_Husband, capital-gain, and engineered features like log_capital_gain, age_x_hours, log_capital_loss, and has_capital_gain, which suggests the feature engineering added real signal beyond raw columns.

The best single model was the tuned Random Forest (test accuracy = 0.8655, CV accuracy = 0.8634), which outperformed tuned Logistic Regression (test accuracy = 0.8588, CV accuracy = 0.8579). The probability-averaging ensemble gave a small improvement (0.8661), so combining models helped, but only but not by that much.

The models did respond differently to engineered features, the non-linear tree model benefited more from interaction and transformed features (for example age_x_hours and log capital features were among high-importance variables), while Logistic Regression remained more constrained by linear decision boundaries and showed lower overall performance, especially on minority-class recall.

In my workflow, this supports a repeatable process: start with baseline models, engineer diverse features (transformations + interactions + grouped indicators), tune each model, compare single-model performance, then test a simple ensemble for incremental gains. I’ll keep using feature importance and class-level metrics to decide which engineered features to keep for future models.